# 🎨 AIGC 电商海报及文案制作 Agent — 多智能体协同生成系统

---

## 项目背景

在中国电商生态中，每天有 **数千万小微商家** 需要持续产出营销素材：商品主图、活动海报、详情页 banner、社交平台种草图……

他们的真实痛点非常具体：

- **没有设计师** — 中小卖家根本养不起一支美术团队，每张图都外包成本高昂
- **不会写文案** — 卖家自己写的文案口语化、缺乏卖点，转化率极低
- **AI 工具门槛高** — Midjourney / Stable Diffusion 需要英文 Prompt 工程能力，普通卖家不会写
- **风格难以统一** — 市面工具往往只解决「画图」或「写字」单一环节，二者割裂导致最终海报「图文不搭」
- **改稿周期长** — 一句模糊需求 → 反复沟通 → 改 5 版才能定稿，时间成本高

## 用户场景

**典型用户画像：** 一位淘宝/抖音小店女装店主，30 岁，没有设计基础，想做一张「夏季新款连衣裙促销海报」。

**她的真实诉求：**

> *「我就想要一张图，能体现夏天的清凉感，价格写上去，让人一看就想买。文案要写得高级一点，别像地摊货。」*

**传统做法：**

1. 自己用 PS 拼图（不会做、效果土）
2. 找美工外包（一张 50~200 元，等 1~3 天）
3. 用模板网站（千篇一律，没有差异化）
4. 直接抄竞品（侵权风险高）

**本系统的做法：**

用户只需用大白话告诉 AI「我想要什么」，**4 个专业智能体**会自动完成：需求澄清 → 卖点提炼 → 文案撰写 → 视觉设计 → 海报渲染 → 质检自检 → 必要时自动重绘。

全流程 **30 秒到 1 分钟** 出图，成本接近于零。

## 需求分析

通过对小微商家、新媒体运营、个人创作者等目标用户的行为分析，提炼出以下核心需求：

| 编号 | 需求描述 | 优先级 | 实现策略 |
|------|---------|--------|---------|
| N1 | 用一句中文白话即可启动，无需懂 Prompt | P0 | Planner Agent 主动澄清缺失要素 |
| N2 | 文案要有「卖点 + 情绪 + 行动呼吁」三段式 | P0 | Copywriter Agent + 行业话术库 |
| N3 | 海报视觉风格要专业、不像 AI 一眼货 | P0 | Art Director Agent + 配色/构图模板库 |
| N4 | 文案与视觉风格要协调统一 | P0 | 多 Agent 共享同一份结构化简报 |
| N5 | 支持上传商品实物图，让海报里的产品就是「我的产品」 | P1 | Visual Agent 启用「图改图」模式 |
| N6 | 出图质量差能自动修正，无需用户操心 | P1 | Visual Agent 内置图像自检 + 自动重绘 |
| N7 | 在中国大陆网络环境下稳定可用 | P0 | 火山引擎模型 + base64 直传，无外部图床依赖 |
| N8 | 全过程透明可追踪，方便用户理解 AI 的「思考」 | P1 | 实时流式日志输出每个 Agent 的工具调用 |

## 解决方案：多智能体（Multi-Agent）协同架构

本项目采用「**专业分工 + 工具加持**」的多智能体设计：每个 Agent 扮演广告公司的一个真实岗位，并配备专属「工具」以避免大模型凭空捏造。

```
          ┌──────────────────────────────────────────┐
          │  用户（中文白话需求 + 可选商品图）        │
          └────────────────┬─────────────────────────┘
                           ↓
          ┌──────────────────────────────────────────┐
          │ 🧠 Planner（项目主控）                    │
          │   工具: parse_brief_to_json              │
          │   职责: 对话澄清 → 触发工作流             │
          └────────────────┬─────────────────────────┘
                           ↓ 结构化简报
          ┌──────────────────────────────────────────┐
          │ ✍️ Copywriter（文案策划）                 │
          │   工具1: keyword_research（行业热词库）   │
          │   工具2: copy_scorer（自评打分器）        │
          └────────────────┬─────────────────────────┘
                           ↓ 三段式文案
          ┌──────────────────────────────────────────┐
          │ 🎨 Art Director（美术指导）               │
          │   工具1: color_palette_designer（配色库） │
          │   工具2: composition_template（构图库）   │
          │   工具3: prompt_structurer（Prompt组装） │
          └────────────────┬─────────────────────────┘
                           ↓ 结构化中文 Prompt
          ┌──────────────────────────────────────────┐
          │ 📸 Visual Agent（视觉渲染）              │
          │   工具1: call_seedream_api（图像生成）    │
          │   工具2: image_quality_check（自检）     │
          │   策略: 自检不过 → 自动改写 Prompt 重绘  │
          └────────────────┬─────────────────────────┘
                           ↓
          ┌──────────────────────────────────────────┐
          │  最终海报 + 文案（PNG + 文本）           │
          └──────────────────────────────────────────┘
```

## 技术架构

```
用户（浏览器 Gradio 界面）
    ↕ 流式对话
Multi-Agent 调度器（本 Notebook · Python）
    ↕ Chat Completions API（OpenAI 兼容）
火山引擎 · Doubao-1.5-pro-32k（文本智能体）
    ↕ Images Generations API（base64 直返）
火山引擎 · Doubao-Seedream（图像渲染）
    ↕ 本地 PNG 落盘
Gradio UI 展示
```

## 功能清单

| 功能 | 说明 |
|------|------|
| 一句话启动 | 中文白话即可，Planner 主动追问缺失要素 |
| 卖点&热词检索 | 内置 6 大品类行业热词库，避免文案空洞 |
| 文案三段式 | 自动产出 主标题 / 副标题 / CTA 行动呼吁 |
| 文案自评打分 | 4 维度评分，<70 分自动重写 |
| 配色专业库 | 7 套商业配色方案，按情绪关键词智能匹配 |
| 构图模板库 | 5 种专业构图法，按品类自动选型 |
| 结构化 Prompt | 把所有视觉决策按图像模型偏好的层级拼装 |
| 图改图模式 | 上传商品实物图，海报里就是你的真品 |
| 图像自检 | 亮度/对比度/分辨率三维度自动质检 |
| 自动重绘 | 质检不过自动改写 Prompt 最多 3 次 |
| 实时流水线 | 每一步工具调用都流式输出，全过程透明 |
| 国内网络友好 | base64 直返，零外部图床依赖 |

---

下面我们一步一步搭建这个系统。

---

## 第一步：安装依赖包

本项目用到的所有 Python 包及用途：

| 包名 | 用途 |
|------|------|
| `gradio` | 一键搭建带聊天框的 Web 界面，自带流式输出能力 |
| `openai` | OpenAI Python SDK；火山引擎接口完全兼容，所以用它即可 |
| `requests` | HTTP 客户端，处理图像下载等通用网络请求 |
| `pillow` | 图像处理库，用于图像质检与本地保存 |
| `numpy<2` | Pillow 与 Gradio 的间接依赖；锁定 1.x 版本以避免兼容性问题 |

> **安装提示**：使用清华镜像源 `pypi.tuna.tsinghua.edu.cn`，在中国大陆访问 PyPI 时下载速度更快、更稳定。

In [1]:
# ============================================================
# 安装项目所需的 Python 包
# 使用清华镜像源 -i https://pypi.tuna.tsinghua.edu.cn/simple
# 在国内网络环境下下载速度更快
# ============================================================

# -q   静默安装，不输出冗长的进度信息
# numpy<2  锁定 numpy 主版本号为 1.x，避免与 Pillow/Gradio 出现 ABI 不兼容
!pip install -q gradio requests pillow openai "numpy<2" -i https://pypi.tuna.tsinghua.edu.cn/simple

print("✅ 所有依赖安装完成")

✅ 所有依赖安装完成


'DOSKEY' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: C:\Python312\python.exe -m pip install --upgrade pip


---

## 第二步：导入必需的 Python 库

把后面会用到的标准库与第三方库一次性导入，方便后续单元格直接调用。

| 模块 | 主要用途 |
|------|---------|
| `os` / `tempfile` / `time` | 路径处理、临时目录、时间戳 |
| `re` | 正则表达式（解析触发口令、抽取主标题等） |
| `json` | 智能体之间结构化数据传递 |
| `base64` / `io.BytesIO` | 图像 Base64 编解码（关键：绕过外部图床） |
| `requests` | 兜底网络请求 |
| `gradio as gr` | UI 框架 |
| `PIL.Image` | 图像处理 |
| `openai.OpenAI` | 大模型客户端（兼容火山引擎） |

In [2]:
# ============================================================
# 集中导入所有第三方库
# 一次性导入有助于：
#   1. 提早暴露环境问题（缺包会立即报错而不是运行到一半）
#   2. 后续代码单元更整洁，无需重复 import
# ============================================================

import os                    # 文件路径与目录操作
import re                    # 正则表达式
import json                  # JSON 编解码（智能体间通信用）
import base64                # Base64 编解码（图像直传 API 用）
import tempfile              # 跨平台临时目录
import time                  # 时间戳，用于生成唯一文件名
from io import BytesIO       # 内存中的二进制缓冲区

import requests              # 通用 HTTP 客户端
import gradio as gr          # Web UI 框架
from PIL import Image        # 图像读取/缩放/编码
from openai import OpenAI    # OpenAI Python SDK（兼容火山引擎接口）

print("✅ 全部库导入成功")

✅ 全部库导入成功


---

## 第三步：配置大模型 API

本项目使用 **火山引擎方舟（Volcengine Ark）** 提供的两类模型：

| 类别 | 模型 | 接入点（Endpoint ID） | 角色 |
|------|------|----------------------|------|
| 文本对话 | Doubao-1.5-pro-32k | `ep-m-20251203181545-krx4h` | 驱动 Planner / Copywriter / Art Director |
| 图像生成 | Doubao-Seedream | `ep-m-20260506225802-v7c99` | 驱动 Visual Agent 的最终渲染 |

火山引擎的 API 完全兼容 OpenAI 协议，所以只要替换 `base_url` 即可继续用 `openai` 库。

> **API Key 说明**：运行前请把火山引擎密钥写入环境变量 `ARK_API_KEY`。Notebook 不内置或提交真实 Key。

In [3]:
# ============================================================
# 大模型 API 配置
# 集中管理所有访问凭据，方便后续替换或迁移到环境变量
# ============================================================

# 火山引擎 API Key（从环境变量读取，不在 Notebook 中保存真实密钥）
DEFAULT_API_KEY = os.getenv("ARK_API_KEY", "").strip()

# 火山引擎方舟 API 基础地址（OpenAI 兼容协议）
DEFAULT_BASE_URL = "https://ark.cn-beijing.volces.com/api/v3"

# 文本模型接入点：Doubao-1.5-pro-32k
# 用于所有文本类智能体（Planner / Copywriter / Art Director）
TEXT_MODEL = "ep-m-20251203181545-krx4h"

# 图像模型接入点：Doubao-Seedream
# 用于 Visual Agent 的最终海报渲染，支持「图改图」模式
IMAGE_MODEL = "ep-m-20260506225802-v7c99"

print("✅ API 配置完成")
print(f"   文本模型: {TEXT_MODEL}")
print(f"   图像模型: {IMAGE_MODEL}")
print(f"   API 地址: {DEFAULT_BASE_URL}")

✅ API 配置完成
   文本模型: ep-m-20251203181545-krx4h
   图像模型: ep-m-20260506225802-v7c99
   API 地址: https://ark.cn-beijing.volces.com/api/v3


---

## 第四步：构建「卖点 & 热词」知识库（Tool 1 的底层数据）

Copywriter Agent 在写文案前，需要先「查资料」——参考行业内已经被市场验证的高转化关键词。

**为什么不让 LLM 直接凭印象写？**

因为 LLM 的训练数据有截止日期，对最新爆款话术（如「显瘦剪裁」「成分党」「松弛感」）的覆盖度参差不齐，凭印象写容易出现「老气」或「空泛」。给它一份结构化的行业词库，等同于给写手一本《行业切片词典》，立刻就能写出更地道的文案。

本项目内置了 **6 大主流品类** 的词库，每个品类包含四个维度：

1. **卖点关键词** — 产品本身的硬性特征（材质、工艺、功能）
2. **情绪关键词** — 用户心理层面的感受（治愈、提神、仪式感）
3. **热门话术模板** — 经过验证的高 CTR 短句
4. **目标人群** — 帮 AI 锁定调性

> 真实生产环境中，这部分可以对接淘宝热搜接口、巨量算数等数据源，实现日级别更新。

In [4]:
# ============================================================
# 卖点&热词知识库（KEYWORD_LIBRARY）
# 结构: { 品类名: { 卖点关键词, 情绪关键词, 热门话术模板, 目标人群 } }
# 覆盖 6 大主流电商品类 + 1 个兜底「通用」类目
# ============================================================

KEYWORD_LIBRARY = {
    # ---------- 咖啡 ----------
    "咖啡": {
        "卖点关键词": ["醇厚回甘", "0蔗糖", "冷萃工艺", "手冲级", "深度烘焙", "意式浓缩", "唤醒晨间"],
        "情绪关键词": ["治愈", "提神", "仪式感", "通勤伴侣", "深夜陪伴", "午后续命"],
        "热门话术模板": ["XX口的清醒", "一杯入魂", "今天也要打起精神", "城市夜归人专属"],
        "目标人群": ["白领", "学生党", "咖啡发烧友", "通勤族"]
    },
    # ---------- 护肤品 ----------
    "护肤品": {
        "卖点关键词": ["补水保湿", "抗老紧致", "敏感肌可用", "成分党", "屏障修护", "提亮肤色"],
        "情绪关键词": ["细腻柔滑", "水润嫩透", "自信光泽", "素颜底气"],
        "热门话术模板": ["养肤如养花", "肌肤会回答", "好状态自己说话", "肌底焕新计划"],
        "目标人群": ["都市白领女性", "成分党", "敏感肌人群", "轻熟龄"]
    },
    # ---------- 服装 ----------
    "服装": {
        "卖点关键词": ["显瘦剪裁", "高级感", "通勤百搭", "高奢质感", "亲肤面料", "版型立体"],
        "情绪关键词": ["松弛感", "氛围感", "都市丽人", "复古优雅", "极简先锋"],
        "热门话术模板": ["穿出你的高级", "通勤也能很自由", "衣橱里的安全感", "一件穿三季"],
        "目标人群": ["都市白领", "时尚买手", "轻熟女性", "Z世代"]
    },
    # ---------- 数码 ----------
    "数码": {
        "卖点关键词": ["旗舰性能", "持久续航", "极致工艺", "降噪科技", "无线自由", "影像旗舰"],
        "情绪关键词": ["科技未来感", "硬核手感", "效率革命", "随心所欲"],
        "热门话术模板": ["重新定义XX", "把性能放进口袋", "为创作者而生", "效率即自由"],
        "目标人群": ["科技爱好者", "学生党", "职场人", "创作者"]
    },
    # ---------- 食品 ----------
    "食品": {
        "卖点关键词": ["真材实料", "0添加", "古法工艺", "原产地直采", "低卡轻负担", "营养均衡"],
        "情绪关键词": ["治愈味蕾", "舌尖记忆", "暖心慰藉", "童年味道"],
        "热门话术模板": ["一口惊艳", "好吃到舔包装", "嘴馋就是这一口", "懂吃的人都在选"],
        "目标人群": ["美食爱好者", "宝妈", "白领", "学生党"]
    },
    # ---------- 兜底通用 ----------
    "通用": {
        "卖点关键词": ["品质优选", "限时特惠", "热销爆款", "新品上市", "口碑之选"],
        "情绪关键词": ["心动", "值得", "美好生活", "专属体验"],
        "热门话术模板": ["懂你所爱", "好物分享", "新品上线", "限时心动价"],
        "目标人群": ["都市消费者", "品质追求者"]
    }
}

print(f"✅ 知识库构建完成，共 {len(KEYWORD_LIBRARY)} 个品类")

✅ 知识库构建完成，共 6 个品类


---

## 第五步：定义「卖点检索」工具（Tool 1 · Copywriter 调用）

上一步搭好了原始词库，本步把它包装成一个**可被 Agent 调用的工具函数**。

**设计要点：**

- 输入用户给的品类（如「冰咖啡」「夏季连衣裙」），做**模糊匹配**——只要包含或被包含都算命中
- 命中失败时返回兜底「通用」库，**绝不返回 None**——保证下游 Agent 永远拿得到有效素材
- 返回字典里多带一个 `matched_category` 字段，方便日志追踪「实际命中了哪个品类」

In [5]:
# ============================================================
# Tool 1: keyword_research
# 用途: 给 Copywriter Agent 提供行业素材，避免空想
# 调用方: Copywriter Agent（在写文案之前必先调一次）
# ============================================================

def keyword_research(category: str) -> dict:
    """卖点 / 热词检索工具。

    参数:
        category: 品类关键词字符串，例如「冰咖啡」「连衣裙」

    返回:
        dict，包含 matched_category（实际命中的品类名）、卖点关键词、
        情绪关键词、热门话术模板、目标人群 五个字段。
    """
    # 防御性处理：去除空白
    category = (category or "").strip()

    # 模糊匹配：只要库的 key 与用户输入相互包含，即视为命中
    for key in KEYWORD_LIBRARY.keys():
        if key in category or category in key:
            return {"matched_category": key, **KEYWORD_LIBRARY[key]}

    # 未命中 → 返回「通用」兜底库，保证 Agent 永不空手
    return {"matched_category": "通用", **KEYWORD_LIBRARY["通用"]}


# 简单自测
_demo = keyword_research("夏日冰咖啡")
print(f"✅ 测试调用: 输入'夏日冰咖啡' → 命中品类「{_demo['matched_category']}」")
print(f"   返回卖点词: {_demo['卖点关键词'][:3]}...")

✅ 测试调用: 输入'夏日冰咖啡' → 命中品类「咖啡」
   返回卖点词: ['醇厚回甘', '0蔗糖', '冷萃工艺']...


---

## 第六步：定义「文案自评打分器」工具（Tool 2 · Copywriter 调用）

Copywriter 写完文案后，让它**自己当裁判**给自己打分。低于 70 分就触发**自动重写**。

**为什么需要自评？**

LLM 一次生成的文案质量参差不齐——有时灵光乍现，有时平平无奇。引入一个 **客观规则评分** 等于在系统中建立「质量底线」，把不合格输出拦在交付前。

**评估维度（4 维各占 25 分）：**

| 维度 | 评分依据 |
|------|---------|
| 长度合规 | 海报文案在 30~200 字之间为佳 |
| 结构完整 | 至少 3 行，对应 主标题/副标题/CTA |
| 抓眼元素 | 含数字、感叹号、强动词（限时/立即/专属…）|
| 简报相关性 | 与原始需求的关键词命中数 ≥2 |

In [6]:
# ============================================================
# Tool 2: copy_scorer
# 用途: 启发式规则给文案打分，触发「不合格则自动重写」机制
# 调用方: Copywriter Agent（写完文案后自评）
# ============================================================

def copy_scorer(copy_text: str, brief: str) -> dict:
    """文案自评打分器（满分 100）。

    参数:
        copy_text: 待评分的文案（通常包含主标题/副标题/CTA 三行）
        brief:     原始创意简报，用来检测关键词相关性

    返回:
        dict { score, feedback, char_count }
    """
    score = 0          # 累计得分
    feedback = []      # 收集需要改进的反馈

    # ----- 维度1: 长度合规性（占 25 分）-----
    # 海报文案不宜过长，30~200 字最佳；过短信息密度不足，过长视觉无法承载
    char_count = len(copy_text)
    if 30 <= char_count <= 200:
        score += 25
    elif char_count < 30:
        feedback.append("文案过短,信息密度不足")
        score += 10
    else:
        feedback.append("文案过长,海报视觉承载力有限,建议精简")
        score += 10

    # ----- 维度2: 结构完整性（占 25 分）-----
    # 标准海报文案应有「主标题 / 副标题 / CTA」三段，至少 3 行
    lines = [l.strip() for l in copy_text.split("\n") if l.strip()]
    if len(lines) >= 3:
        score += 25
    else:
        feedback.append("结构层次不够,建议明确区分主标题、副标题与行动呼吁")
        score += 10

    # ----- 维度3: 抓眼元素（占 25 分）-----
    # 数字 / 感叹号 / 限时类强转化词数量 ≥2 视为达标
    catchy_signals = re.findall(
        r'[0-9]+|[!！?？]|立即|马上|现在|限时|首发|专属|爆款',
        copy_text
    )
    if len(catchy_signals) >= 2:
        score += 25
    else:
        feedback.append("缺乏抓眼元素,可加入数字、感叹号或限时类强转化词")
        score += 10

    # ----- 维度4: 与简报的相关性（占 25 分）-----
    # 提取简报中的中文关键词（2~4 字），统计文案中命中数
    brief_keywords = re.findall(r'[\u4e00-\u9fa5]{2,4}', brief or "")
    hit = sum(1 for kw in set(brief_keywords) if kw in copy_text)
    if hit >= 2:
        score += 25
    else:
        feedback.append("与简报核心关键词关联度偏低")
        score += 10

    return {
        "score": score,
        "feedback": feedback if feedback else ["整体表现优秀"],
        "char_count": char_count
    }

print("✅ Tool 2 文案打分器定义完成")

✅ Tool 2 文案打分器定义完成


---

## 第七步：构建「配色方案库」（Tool 3 的底层数据）

Art Director 不能让 AI 自由配色，那样很容易出现土味红绿配。

本步内置 **7 套商业级配色方案**，每套包含：

| 字段 | 含义 |
|------|------|
| `primary` | 主色（占画面 60%） |
| `secondary` | 辅色（占 30%） |
| `accent` | 点缀色（占 10%） |
| `mood` | 情绪标签（驱动后续匹配规则） |
| `rgb_desc` | 给 AI 看的中文描述 |

7 套方案覆盖了夏日、奢华、治愈、潮酷等几乎全部主流电商场景。

In [7]:
# ============================================================
# 配色方案库（COLOR_PALETTES）
# 7 套预设：每套含 主色 / 辅色 / 点缀色 / 情绪 / 中文描述
# ============================================================

COLOR_PALETTES = {
    "夏日清凉": {
        "primary": "#4FC3F7", "secondary": "#FFFFFF", "accent": "#FFEB3B",
        "mood": "清爽、明亮、活力",
        "rgb_desc": "天蓝主调、纯白留白、柠檬黄点缀"
    },
    "高级冷感": {
        "primary": "#1A237E", "secondary": "#37474F", "accent": "#C0C0C0",
        "mood": "冷峻、专业、科技",
        "rgb_desc": "深海蓝主调、石墨灰辅助、银色点缀"
    },
    "暖心治愈": {
        "primary": "#FF8A65", "secondary": "#FFE0B2", "accent": "#8D6E63",
        "mood": "温暖、舒适、亲和",
        "rgb_desc": "珊瑚橙主调、奶油米辅助、可可棕点缀"
    },
    "高奢质感": {
        "primary": "#212121", "secondary": "#D4AF37", "accent": "#FFFFFF",
        "mood": "尊贵、典雅、稀缺",
        "rgb_desc": "墨黑主调、香槟金辅助、纯白点缀"
    },
    "自然清新": {
        "primary": "#81C784", "secondary": "#F1F8E9", "accent": "#FFB300",
        "mood": "天然、纯粹、生机",
        "rgb_desc": "薄荷绿主调、米白辅助、暖黄点缀"
    },
    "Z世代潮酷": {
        "primary": "#E91E63", "secondary": "#9C27B0", "accent": "#00E5FF",
        "mood": "张扬、潮流、个性",
        "rgb_desc": "玫红主调、紫调辅助、电光蓝点缀"
    },
    "极简北欧": {
        "primary": "#FAFAFA", "secondary": "#BDBDBD", "accent": "#FF5252",
        "mood": "简约、纯净、克制",
        "rgb_desc": "雪白主调、浅灰辅助、亮红点缀"
    }
}

print(f"✅ 配色方案库构建完成，共 {len(COLOR_PALETTES)} 套预设")

✅ 配色方案库构建完成，共 7 套预设


---

## 第八步：定义「配色方案设计器」工具（Tool 3 · Art Director 调用）

把上一步的配色库包装成可调用的工具。**核心是一个关键词→配色名的映射规则表**。

Art Director 调用时只需传入一段「情绪/风格描述」，工具自动匹配最贴近的预设方案。

> 这种「LLM 出关键词 + 规则匹配硬编码方案」的混合架构，比让 LLM 自己写十六进制 RGB 值要稳定得多。

In [8]:
# ============================================================
# Tool 3: color_palette_designer
# 用途: 把模糊的「情绪/风格关键词」映射到具体的专业配色方案
# 调用方: Art Director Agent
# ============================================================

def color_palette_designer(mood_keywords: str) -> dict:
    """配色方案设计器。

    参数:
        mood_keywords: 自由文本的情绪/风格描述
                       例如「夏日清凉」「科技未来感」「高奢质感」

    返回:
        dict，含命中方案的 name / primary / secondary / accent / mood / rgb_desc
    """
    # 统一小写做匹配，提高鲁棒性
    mood_keywords = (mood_keywords or "").lower()

    # 关键词触发规则表：(触发词列表, 对应配色方案名)
    # 顺序决定优先级——越靠前的规则越优先匹配
    rules = [
        (["冷", "夏日", "清凉", "冰", "海", "蓝"],          "夏日清凉"),
        (["科技", "未来", "高级", "数码", "专业", "冷峻"], "高级冷感"),
        (["温暖", "治愈", "暖心", "亲和", "家", "陪伴"],   "暖心治愈"),
        (["奢华", "高端", "尊贵", "精致", "高奢", "金"],   "高奢质感"),
        (["自然", "清新", "绿色", "健康", "有机", "户外"], "自然清新"),
        (["潮流", "z世代", "年轻", "潮酷", "炫酷", "个性"], "Z世代潮酷"),
        (["极简", "北欧", "简约", "纯净", "克制", "性冷淡"], "极简北欧"),
    ]

    # 遍历规则，命中即返回
    for triggers, palette_name in rules:
        if any(t in mood_keywords for t in triggers):
            return {"name": palette_name, **COLOR_PALETTES[palette_name]}

    # 兜底：默认极简北欧（百搭、最不容易出错的方案）
    return {"name": "极简北欧", **COLOR_PALETTES["极简北欧"]}

# 自测
_p = color_palette_designer("夏日清凉")
print(f"✅ 测试调用: 输入'夏日清凉' → 命中「{_p['name']}」, 主色 {_p['primary']}")

✅ 测试调用: 输入'夏日清凉' → 命中「夏日清凉」, 主色 #4FC3F7


---

## 第九步：构建「构图模板库」（Tool 4 的底层数据）

构图是海报视觉的骨架。不同品类天然适合不同构图：

| 构图法 | 典型场景 |
|--------|---------|
| 中心特写式 | 数码新品、单品突显 |
| 场景沉浸式 | 服装、家居、生活方式 |
| 悬浮飞溅式 | 饮品、化妆品、运动 |
| 对称仪式感式 | 高奢、礼品、节日营销 |
| 九宫格三分式 | 通用商业海报 |

每种模板携带：

1. `scene_layout` — 画面元素如何排布
2. `text_zone` — 给文案预留哪块安全区（关键！否则文字会被产品挡住）
3. `lens` — 镜头参数（焦距/景深/光线）
4. `fit_for` — 适用场景

这些字段会被原样喂给图像模型，作为高质量提示词的一部分。

In [9]:
# ============================================================
# 构图模板库（COMPOSITION_TEMPLATES）
# 5 种专业构图法，每种含布局/文字安全区/镜头参数/适用场景
# ============================================================

COMPOSITION_TEMPLATES = {
    # ---------- 中心特写式 ----------
    "中心特写式": {
        "scene_layout": "产品居中超大特写,占据画面 60% 视觉重心,背景虚化弱化干扰",
        "text_zone":    "上方 1/4 留白排主标题,下方 1/4 留白放副标题与 CTA",
        "lens":         "微距镜头、浅景深、f/1.8 大光圈虚化",
        "fit_for":      "新品发布、单品突显、质感产品"
    },
    # ---------- 场景沉浸式 ----------
    "场景沉浸式": {
        "scene_layout": "产品融入真实使用场景,环境元素叙事化,人物或环境占主导",
        "text_zone":    "右上角或左下角负空间预留排版区",
        "lens":         "中焦镜头、自然光、真实环境感",
        "fit_for":      "生活方式、情感营销、场景化卖点"
    },
    # ---------- 悬浮飞溅式 ----------
    "悬浮飞溅式": {
        "scene_layout": "产品悬浮于画面中心,周围环绕动态元素(水花/粉末/光斑)放射状散开",
        "text_zone":    "上下两端各留 1/5 安全排版区",
        "lens":         "高速快门冻结瞬间、戏剧打光、强调动感",
        "fit_for":      "饮品、化妆品、运动产品"
    },
    # ---------- 对称仪式感式 ----------
    "对称仪式感式": {
        "scene_layout": "严格对称构图,产品居中,左右元素镜像呼应,极强秩序感",
        "text_zone":    "顶部居中放主标题,底部居中放 CTA",
        "lens":         "正面平视、均衡布光、消除透视畸变",
        "fit_for":      "高奢产品、礼品、节日营销"
    },
    # ---------- 九宫格三分式（万能兜底）----------
    "九宫格三分式": {
        "scene_layout": "产品置于三分线交点,背景延展形成画面纵深,符合古典美学",
        "text_zone":    "对角象限留白,大面积负空间承载文案",
        "lens":         "中焦平视、自然透视、柔和环境光",
        "fit_for":      "通用商业海报、复合卖点表达"
    }
}

print(f"✅ 构图模板库构建完成，共 {len(COMPOSITION_TEMPLATES)} 种专业构图")

✅ 构图模板库构建完成，共 5 种专业构图


---

## 第十步：定义「构图模板选型」工具（Tool 4 · Art Director 调用）

和配色一样，把「品类 + 风格关键词」映射到具体构图模板。

**匹配优先级（从高到低）：**

1. 饮品 / 化妆品 / 运动 → 悬浮飞溅式（最有冲击力）
2. 高奢 / 礼品 → 对称仪式感式
3. 服装 / 家居 / 生活方式 → 场景沉浸式
4. 数码 / 新品 / 单品 → 中心特写式
5. 其他 → 九宫格三分式（万能兜底）

In [10]:
# ============================================================
# Tool 4: composition_template
# 用途: 根据品类+风格自动匹配最合适的构图模板
# 调用方: Art Director Agent
# ============================================================

def composition_template(product_type: str, style_hint: str) -> dict:
    """构图模板选型工具。

    参数:
        product_type: 产品品类（如「咖啡」「连衣裙」）
        style_hint:   风格倾向（如「夏日清凉」「高奢」）

    返回:
        dict，含命中模板的 name / scene_layout / text_zone / lens / fit_for
    """
    # 把品类与风格拼成一个组合字符串，统一小写匹配
    combined = f"{product_type} {style_hint}".lower()

    # 触发规则表（优先级从上到下）
    rules = [
        (["饮品", "饮料", "咖啡", "化妆", "护肤", "运动", "动感", "飞溅"], "悬浮飞溅式"),
        (["高奢", "奢华", "礼品", "节日", "对称", "仪式"],                  "对称仪式感式"),
        (["生活", "场景", "情感", "服装", "家居", "沉浸"],                  "场景沉浸式"),
        (["新品", "单品", "数码", "特写", "质感"],                          "中心特写式"),
    ]

    for triggers, tpl_name in rules:
        if any(t in combined for t in triggers):
            return {"name": tpl_name, **COMPOSITION_TEMPLATES[tpl_name]}

    # 兜底：万能的九宫格三分式
    return {"name": "九宫格三分式", **COMPOSITION_TEMPLATES["九宫格三分式"]}

# 自测
_c = composition_template("咖啡", "夏日清凉")
print(f"✅ 测试调用: 咖啡+夏日清凉 → 推荐「{_c['name']}」")

✅ 测试调用: 咖啡+夏日清凉 → 推荐「悬浮飞溅式」


---

## 第十一步：定义「结构化 Prompt 组装器」工具（Tool 5 · Art Director 调用）

Art Director 完成所有视觉决策后，需要把它们**按图像模型偏好的格式拼接成一段最终 Prompt**。

**为什么不直接让 LLM 写自由文本 Prompt？**

经过实测，Doubao-Seedream / Stable Diffusion 等图像模型对「**结构化、带显式标签、含数值参数**」的 Prompt 表现远优于自由散文。本工具用统一的中文标签（【主体】【场景与环境】【构图】…）固化输出，显著提升出图稳定性。

**特别处理：画面文字嵌入**

图像模型对长中文文字渲染普遍不稳定，因此本工具只**抽取主标题（≤20 字）**嵌入画面，其余文案靠后期排版叠加。

In [11]:
# ============================================================
# Tool 5: prompt_structurer
# 用途: 把分散的视觉决策按图像模型偏好的格式拼成最终 Prompt
# 调用方: Art Director Agent（视觉链路的最后一步）
# ============================================================

def prompt_structurer(subject: str, environment: str, palette: dict,
                      composition: dict, copy_for_poster: str = "",
                      extra_quality: str = "") -> str:
    """结构化 Prompt 组装器。

    参数:
        subject:         主体描述（产品/人物的外观）
        environment:     环境/场景描述
        palette:         color_palette_designer 返回的配色字典
        composition:     composition_template 返回的构图字典
        copy_for_poster: 已定稿文案（用于抽取主标题嵌入画面）
        extra_quality:   额外质量增强词（默认值已含商业级关键词）

    返回:
        str，最终交付给图像模型的中文 Prompt
    """
    # 默认质量标签——参考社区高频商业级关键词组合
    quality_tag = extra_quality or (
        "8K超高清, 商业摄影级质感, masterpiece, best quality, ultra detailed, 锐利对焦"
    )

    # 按【中文标签】+ 内容的格式组装，每段一行
    parts = [
        f"【主体】{subject}",
        f"【场景与环境】{environment}",
        f"【构图】{composition.get('scene_layout', '')};镜头:{composition.get('lens', '')}",
        f"【文字排版安全区】{composition.get('text_zone', '')}",
        f"【配色方案】{palette.get('name', '')}风格:主色 {palette.get('primary', '')}"
        f"({palette.get('rgb_desc', '')}),整体氛围呈现{palette.get('mood', '')}",
        f"【光影】专业商业布光,主光柔和明确,辅光勾勒轮廓,强化产品立体质感",
        f"【画质】{quality_tag}",
    ]

    # 处理画面文字嵌入
    if copy_for_poster:
        # 优先识别「主标题: xxx」格式；否则取首行
        title_match = re.search(r'主标题[::]\s*(.+)', copy_for_poster)
        if title_match:
            first_line = title_match.group(1).strip()[:20]
        else:
            first_line = copy_for_poster.strip().split("\n")[0][:20]
        parts.append(
            f'【画面文字】在文字安全区内清晰呈现中文标题"{first_line}",'
            f'字体设计感强、可读性高'
        )

    # 用「;\n」连接，提升模型对结构的识别度
    return ";\n".join(parts)

print("✅ Tool 5 Prompt 组装器定义完成")

✅ Tool 5 Prompt 组装器定义完成


---

## 第十二步：定义「图像质检」工具（Tool 6 · Visual Agent 调用）

Visual Agent 出图后，**自动质检**当前生成的图像是否合格。不合格则改写 Prompt 重绘（最多 3 次）。

**质检维度：**

| 维度 | 判定 | 扣分 |
|------|------|------|
| 分辨率 | < 512×512 视为不合格 | -30 |
| 平均亮度 | < 30（过暗）或 > 230（过曝） | -25 |
| 对比度方差 | < 100（画面单调）| -30 |

≥ 70 分视为通过。

**关键设计：零网络依赖**

本工具直接读取本地 PNG 文件做检查，**不依赖任何外部图床域名**。这是为了在中国大陆任意网络环境下都能稳定运行。

In [12]:
# ============================================================
# Tool 6: image_quality_check
# 用途: 对生成的海报做客观质量检查，触发自动重绘机制
# 调用方: Visual Agent
# ============================================================

def image_quality_check(image_path: str) -> dict:
    """图像质量自检。

    参数:
        image_path: 本地图像路径（主路径） 或 URL（兼容降级路径）

    返回:
        dict { pass, score, issues, suggestions, size, brightness }
    """
    # 边界条件: 路径为空
    if not image_path:
        return {"pass": False, "score": 0,
                "issues": ["未生成图像文件"],
                "suggestions": ["重试图像生成"]}

    # 兼容旧版 API 返回 URL 的情况
    is_url = isinstance(image_path, str) and image_path.startswith(("http://", "https://"))

    try:
        # ----- 加载图像（优先本地，URL 仅作降级）-----
        if is_url:
            resp = requests.get(image_path, timeout=20)
            if resp.status_code != 200:
                return {"pass": False, "score": 20,
                        "issues": [f"图像下载失败 HTTP {resp.status_code}"],
                        "suggestions": ["重试图像生成"]}
            img = Image.open(BytesIO(resp.content))
        else:
            if not os.path.exists(image_path):
                return {"pass": False, "score": 0,
                        "issues": [f"本地图像文件不存在: {image_path}"],
                        "suggestions": ["重试图像生成"]}
            img = Image.open(image_path)

        w, h = img.size

        # 累计扣分体系：满分 100，不达标项依次扣分
        issues = []
        suggestions = []
        score = 100

        # ----- 检查1: 分辨率 -----
        if w < 512 or h < 512:
            issues.append(f"分辨率偏低 {w}x{h}")
            suggestions.append("提高生成分辨率")
            score -= 30

        # ----- 检查2: 平均亮度 -----
        # 转灰度 → 缩略 64x64 提速 → 算像素均值
        gray = img.convert("L")
        thumb = gray.resize((64, 64))
        pixels = list(thumb.getdata())
        mean_brightness = sum(pixels) / len(pixels)

        if mean_brightness < 30:
            issues.append(f"画面整体过暗 (亮度={mean_brightness:.0f})")
            suggestions.append("Prompt 中加入'明亮布光、提亮主体'")
            score -= 25
        elif mean_brightness > 230:
            issues.append(f"画面整体过曝 (亮度={mean_brightness:.0f})")
            suggestions.append("Prompt 中加入'柔和布光、避免过曝'")
            score -= 25

        # ----- 检查3: 对比度（方差） -----
        # 方差小 ≈ 像素值集中 ≈ 画面单调（极端情况下可能是纯色异常输出）
        variance = sum((p - mean_brightness) ** 2 for p in pixels) / len(pixels)
        if variance < 100:
            issues.append("画面对比度过低,可能内容贫乏")
            suggestions.append("Prompt 中强化'丰富层次、强对比、立体感'")
            score -= 30

        return {
            "pass": score >= 70,
            "score": score,
            "issues": issues if issues else ["未检出明显问题"],
            "suggestions": suggestions if suggestions else ["质量达标,无需改动"],
            "size": f"{w}x{h}",
            "brightness": round(mean_brightness, 1)
        }
    except Exception as e:
        # 异常兜底：永不抛出，永远返回结构化失败
        return {"pass": False, "score": 0,
                "issues": [f"自检异常: {e}"],
                "suggestions": ["重试图像生成"]}

print("✅ Tool 6 图像质检工具定义完成")

✅ Tool 6 图像质检工具定义完成


---

## 第十三步：定义「简报结构化解析器」工具（Tool 7 · Planner 调用）

Planner 与用户对话后，会给出一段自然语言的「执行简报」。

如果直接把这段文字塞给下游 Copywriter / Art Director，他们都得**重新理解一遍语义**——既浪费 Token 又容易理解不一致。

本工具利用 **LLM 自身做信息抽取**，把简报转为统一的 JSON：

```json
{
  "product_category": "咖啡",
  "product_name": "夏日冷萃",
  "core_selling_points": ["0蔗糖", "冷萃工艺"],
  "target_audience": "都市白领",
  "tone": "清新有活力",
  "visual_style": "夏日清凉",
  "scene_hint": "冰块与水珠飞溅特写",
  "must_include_text": "夏日冷萃"
}
```

下游 Agent 拿到结构化数据后，可直接索引字段，无需自然语言理解。

In [13]:
# ============================================================
# Tool 7: parse_brief_to_json
# 用途: 把 Planner 的自然语言简报转为结构化 JSON
# 调用方: Planner Agent（在触发工作流后立刻调）
# ============================================================

def parse_brief_to_json(planner_text: str, llm_client, text_model) -> dict:
    """利用 LLM 做信息抽取，输出结构化 JSON。

    参数:
        planner_text: Planner 给出的自然语言简报
        llm_client:   已初始化的 OpenAI 客户端
        text_model:   文本模型接入点

    返回:
        dict，标准化字段。即便解析失败也返回带兜底字段的 dict
    """
    # 强约束的抽取提示词：明确 JSON 字段、禁止 markdown 包裹
    extract_prompt = """你是一个严格的信息抽取器。从下面的创意简报中,抽取关键字段,只输出 JSON,无任何额外文字、不要 ```代码块标记。

需要抽取的字段:
{
  "product_category": "产品品类(如:咖啡、护肤品、服装、数码、食品)",
  "product_name": "具体产品名称",
  "core_selling_points": ["卖点1", "卖点2"],
  "target_audience": "目标人群",
  "tone": "文案调性(如:温暖治愈/科技冷感/年轻活力)",
  "visual_style": "视觉风格(如:夏日清凉/高奢质感/极简北欧)",
  "scene_hint": "场景提示",
  "must_include_text": "必须出现在画面上的文字(可空)"
}

简报原文:
"""
    try:
        resp = llm_client.chat.completions.create(
            model=text_model,
            messages=[
                {"role": "system", "content": extract_prompt},
                {"role": "user",   "content": planner_text}
            ]
        )
        raw = resp.choices[0].message.content.strip()

        # 兜底清洗：即便提示词禁止了 ```，模型偶尔仍会加，统一剥掉
        raw = re.sub(r'^```(?:json)?\s*', '', raw)
        raw = re.sub(r'\s*```$', '', raw)
        return json.loads(raw)

    except Exception as e:
        # 解析失败时返回空骨架 + 原始文本，下游仍可用自然语言原文兜底
        return {
            "product_category":    "通用",
            "product_name":        "",
            "core_selling_points": [],
            "target_audience":     "",
            "tone":                "",
            "visual_style":        "",
            "scene_hint":          "",
            "must_include_text":   "",
            "_parse_error":        str(e),
            "_raw":                planner_text
        }

print("✅ Tool 7 简报结构化解析器定义完成")

✅ Tool 7 简报结构化解析器定义完成


---

## 第十四步：Planner 智能体提示词

Planner 是整个系统的**项目主控**，负责：

1. 与用户对话，澄清模糊需求
2. 判断需求是否已经完整
3. 完整后输出特定的「触发口令」启动下游工作流

**触发口令格式：**

```
[EXECUTE_WORKFLOW: 产品品类:xxx;产品名:xxx;核心卖点:xxx;...]
```

系统通过正则识别这个口令并提取简报内容，启动下游 Agent。这是一种典型的 **ReAct（Reasoning + Acting）** 路由模式。

In [14]:
# ============================================================
# PLANNER_PROMPT
# Planner Agent 的系统提示词，定义其角色、职责与触发口令格式
# ============================================================

PLANNER_PROMPT = """你是由多智能体协同架构驱动的「AIGC项目主控(Planner)」。你的核心任务是与用户沟通、精准收集需求,并统筹下游的文案(Copywriter)和美术(Art Director)团队。

【职责与运行机制】:
1. 需求澄清与引导:当用户需求模糊、缺失关键要素(如:产品核心卖点、受众、视觉风格等)时,应主动且专业地向用户提问,引导其完善需求细节。
2. 自然对话交互:面对用户的反馈、修改意见或闲聊,应保持专业、礼貌的态度进行评估,并用自然语言进行解答。
3. 自动化工作流触发:当你判定用户意图已明确,且各项必需材料已就绪(如用户表示"可以开始"、"直接生成"、"按这个重新画"等),你必须在回复的末尾追加特定的触发口令指令。

【触发口令格式规范】:
[EXECUTE_WORKFLOW: <生成一份结构化、详尽的任务执行简报,包含:产品品类、产品名、核心卖点、目标人群、文案调性、视觉风格、场景提示、必须出现的文字>]

(例如:[EXECUTE_WORKFLOW: 产品品类:咖啡;产品名:夏日冷萃;核心卖点:0蔗糖、冷萃工艺、清爽提神;目标人群:都市白领;文案调性:清新有活力;视觉风格:夏日清凉;场景提示:冰块与水珠飞溅特写;必须出现的文字:夏日冷萃])

【注意事项】:
- 简报字段越完整,下游团队产出质量越高(下游会自动调用结构化解析器读取你的简报)。
- 若决定触发工作流,向用户展示的对话内容请尽量简练干脆。
"""

print(f"✅ PLANNER_PROMPT 定义完成 ({len(PLANNER_PROMPT)} 字符)")

✅ PLANNER_PROMPT 定义完成 (619 字符)


---

## 第十五步：Copywriter 智能体提示词模板

Copywriter 不是凭空写文案，而是**基于 Tool 1（卖点检索）的输出**做创作。

提示词中的 `{keyword_research_result}` 是占位符，运行时会被实际的工具调用结果替换进去。

**输出格式严格约束为三行：**

```
主标题: <主标题文字>
副标题: <副标题文字>
CTA: <行动呼吁文字>
```

这种「**LLM 创作 + 严格格式 + 工具上下文**」的组合，能稳定产出可用文案。

In [15]:
# ============================================================
# COPYWRITER_PROMPT_TEMPLATE
# Copywriter Agent 的系统提示词模板，{keyword_research_result} 运行时填充
# ============================================================

COPYWRITER_PROMPT_TEMPLATE = """你是一位屡获殊荣的「资深商业广告文案(Copywriter Agent)」。你拥有以下已为你预先调用好的工具结果,请充分利用它们而不要凭空创作:

【工具1 输出 - 行业卖点/热词参考库】:
{keyword_research_result}

【创作准则】:
1. 优先复用工具返回的卖点关键词、情绪词、热门话术模板——这些是经过市场验证的高转化元素。
2. 结构必须三段式:主标题(8字内、抓眼)/副标题(15字内、补充卖点)/CTA行动呼吁(8字内、强转化)。
3. 文案需嵌入数字、限时、专属等强转化词以提高 CTR。
4. 直接交付最高质量的文案正文,绝不要输出诸如"好的,这是您的文案"等废话或任何标注。

输出格式严格如下(三行,无其他内容):
主标题: <主标题文字>
副标题: <副标题文字>
CTA: <行动呼吁文字>
"""

print(f"✅ COPYWRITER_PROMPT_TEMPLATE 定义完成 ({len(COPYWRITER_PROMPT_TEMPLATE)} 字符)")

✅ COPYWRITER_PROMPT_TEMPLATE 定义完成 (371 字符)


---

## 第十六步：Art Director 智能体提示词模板

Art Director 的提示词模板有 4 个占位符：

| 占位符 | 来源 |
|--------|------|
| `{palette_result}` | Tool 3（配色设计器）输出 |
| `{composition_result}` | Tool 4（构图选型）输出 |
| `{copy_result}` | Copywriter 已定稿文案 |
| `{has_reference_image}` | 用户是否上传了商品参考图（影响主体描述策略）|

**关键差异化设计：**

如果用户上传了商品参考图，Art Director **不要描写商品外观**——因为 Seedream 会以参考图为准还原商品本体。Art Director 此时把全部精力投入「环境/场景」描述。

如果没有参考图，则正常详尽描写主体。

In [16]:
# ============================================================
# ART_DIRECTOR_PROMPT_TEMPLATE
# Art Director Agent 的系统提示词模板，4 个占位符运行时填充
# ============================================================

ART_DIRECTOR_PROMPT_TEMPLATE = """你是一位享誉业界的「资深美术指导(Art Director)兼顶尖 AI 绘画提示词工程专家」。你拥有以下已为你预先调用好的视觉决策工具结果,你的任务是把它们融合成一段最终交付给图像模型的中文 Prompt:

【工具1 输出 - 配色方案】:
{palette_result}

【工具2 输出 - 构图模板】:
{composition_result}

【已定稿文案】:
{copy_result}

【是否有商品参考图】:
{has_reference_image}

【任务】:
请基于上述工具输出,围绕"主体描述"和"环境/场景描述"两块进行细化创作,并保证最终结果体现出工具给出的配色与构图。注意:
- 描述要具体、有画面感,避免空泛形容词。
- 为画面文字预留出工具指定的安全排版区。
- 若文案中有需要嵌入画面的标题文字,请以中文双引号"..."明确标出,字体要求设计感+高可读性。
- ⚠️ 若上方"是否有商品参考图"为「有」: 主体描述中**不要**虚构产品外观、材质、形态、Logo、配色,只描述商品的姿态、角度、与场景的关系——因为模型会直接以参考图为准还原商品本体。重点投入在环境描述上。
- ⚠️ 若上方"是否有商品参考图"为「无」: 正常详尽描述主体外观、材质、形态。

只输出两段中文文字,严格按以下格式(两行):
主体: <对产品/人物的具体外观、材质、姿态描写>
环境: <对场景、背景、氛围、关键道具的具体描写>
"""

print(f"✅ ART_DIRECTOR_PROMPT_TEMPLATE 定义完成 ({len(ART_DIRECTOR_PROMPT_TEMPLATE)} 字符)")

✅ ART_DIRECTOR_PROMPT_TEMPLATE 定义完成 (627 字符)


---

## 第十七步：图像 Base64 编码工具

「图改图」模式下，需要把用户上传的商品图随 API 请求一起传给 Seedream。

**为什么不直接传图床 URL？**

- 用户上传的图在本地，没有公网 URL
- 自建图床增加复杂度
- 国内访问境外图床有不稳定风险

**最稳定的做法：把图片编码为 Base64 data URI**，与 prompt 一起放进请求体，零外部依赖。

**性能优化：**

- 长边超过 1024px 自动等比缩放（Seedream 参考图无需更高分辨率）
- 统一编码为 JPEG（体积比 PNG 小很多）
- 体积仍 > 4MB 时逐步降低 JPEG 质量

In [17]:
# ============================================================
# 本地图像输出目录 + 图像→base64 转换工具
# ============================================================

# 海报渲染结果统一保存到系统临时目录的子目录
# 用 tempfile.gettempdir() 跨平台兼容（Windows/Linux/macOS）
LOCAL_IMAGE_DIR = os.path.join(tempfile.gettempdir(), "aigc_poster_outputs")
os.makedirs(LOCAL_IMAGE_DIR, exist_ok=True)


def _image_to_data_uri(image_path: str, max_side: int = 1024,
                       jpeg_quality: int = 85) -> tuple:
    """把本地图像编码为 base64 data URI，可直接嵌入 API 请求体。

    参数:
        image_path:   本地图像路径
        max_side:     长边缩放上限（默认 1024，对参考图够用）
        jpeg_quality: JPEG 编码质量初始值（默认 85）

    返回:
        (data_uri, info_msg)
        失败时 data_uri 为 None，info_msg 含失败原因
    """
    # 边界条件
    if not image_path:
        return None, "未提供图片路径"
    if not os.path.exists(image_path):
        return None, f"图片不存在: {image_path}"

    try:
        img = Image.open(image_path)

        # ----- EXIF 方向修正 -----
        # 部分手机拍摄的 JPG 自带 EXIF 旋转标记，不修正会导致图片倒置
        try:
            from PIL import ImageOps
            img = ImageOps.exif_transpose(img)
        except Exception:
            pass

        # ----- 模式转换 -----
        # RGBA / P 模式不能直接保存为 JPEG，统一转 RGB
        if img.mode not in ("RGB", "L"):
            img = img.convert("RGB")

        # ----- 等比缩放到长边 max_side -----
        w, h = img.size
        original_size = f"{w}x{h}"
        if max(w, h) > max_side:
            scale = max_side / max(w, h)
            new_size = (int(w * scale), int(h * scale))
            img = img.resize(new_size, Image.LANCZOS)  # LANCZOS 是高质量缩放算法

        # ----- 编码为 JPEG 字节流 -----
        buf = BytesIO()
        img.save(buf, format="JPEG", quality=jpeg_quality, optimize=True)
        img_bytes = buf.getvalue()

        # ----- 体积控制 -----
        # 若仍 > 4MB，逐步降低质量直到 ≤ 4MB 或质量 ≤ 40
        q = jpeg_quality
        while len(img_bytes) > 4 * 1024 * 1024 and q > 40:
            q -= 15
            buf = BytesIO()
            img.save(buf, format="JPEG", quality=q, optimize=True)
            img_bytes = buf.getvalue()

        # ----- 拼装 data URI -----
        b64 = base64.b64encode(img_bytes).decode("utf-8")
        size_kb = len(img_bytes) / 1024
        data_uri = f"data:image/jpeg;base64,{b64}"

        return data_uri, f"原图 {original_size} → 压缩后 {img.size[0]}x{img.size[1]}, {size_kb:.0f}KB"

    except Exception as e:
        return None, f"图像预处理异常: {e}"

print(f"✅ 图像编码工具定义完成；本地输出目录: {LOCAL_IMAGE_DIR}")

✅ 图像编码工具定义完成；本地输出目录: C:\Users\86139\AppData\Local\Temp\aigc_poster_outputs


---

## 第十八步：图像生成 API 调用（Visual Agent 核心）

Visual Agent 的核心动作就是调用 Doubao-Seedream 的 Images Generations 接口。

**关键设计决策：**

| 决策点 | 方案 | 原因 |
|--------|------|------|
| 响应格式 | `response_format="b64_json"` | 直接返回 base64，无需访问境外图床 |
| 「图改图」模式 | `extra_body["image"]` 传 data URI 数组 | 让 Seedream 以商品实物为主体 |
| 多图融合 | `sequential_image_generation="disabled"` | 多参考图时强制单图输出 |
| 出图分辨率 | `1440x2560`（9:16 竖图） | 适合电商主图与小红书 |
| 异常透传 | 抓取 `e.response.text` 等详细字段 | UI 可显示根因 |

如果 b64_json 异常为空，会**降级到 URL 模式**作为兜底。

In [18]:
# ============================================================
# call_seedream_api: Visual Agent 的核心动作 —— 调用 Doubao-Seedream
# ============================================================

def call_seedream_api(api_key: str, base_url: str, prompt: str,
                      reference_images: list = None) -> tuple:
    """调用 Seedream 生成海报。

    参数:
        api_key:          火山引擎 API Key
        base_url:         API 基础地址
        prompt:           最终的图像生成提示词
        reference_images: 商品参考图本地路径列表（启用「图改图」模式）

    返回:
        (local_path, info_or_error_msg)
        成功: (本地图片路径, "渲染成功 + 含可选预处理日志")
        失败: (None,         "失败原因详情")
    """
    info_log = []
    try:
        # 显式拉长 timeout——图像生成耗时较长（30~60s）
        client = OpenAI(api_key=api_key, base_url=base_url, timeout=180.0)

        # ----- 标准 OpenAI 兼容参数 -----
        std_kwargs = {
            "model": IMAGE_MODEL,
            "prompt": prompt,
            "size": "1440x2560",          # 9:16 竖图，适配电商场景
            "response_format": "b64_json", # ⭐ 关键: base64 直返
        }

        # ----- 非标准参数走 extra_body -----
        # ⚠️ 重要修复: image / sequential_image_generation 不是 OpenAI 标准参数
        # 必须通过 extra_body 透传，否则 SDK 会抛 "unexpected keyword argument"
        extra_body = {}

        # ----- 处理参考图（图改图模式）-----
        if reference_images:
            data_uris = []
            for img_path in reference_images:
                duri, msg = _image_to_data_uri(img_path)
                if duri:
                    data_uris.append(duri)
                    info_log.append(f"参考图 {os.path.basename(img_path)} | {msg}")
                else:
                    info_log.append(f"⚠️ 参考图 {os.path.basename(img_path)} 处理失败: {msg}")
            if data_uris:
                # Seedream 4.0+ 支持 image 参数：URL 或 base64 data URI 数组（最多 10 张）
                extra_body["image"] = data_uris
                # 多图融合时关闭分镜组图
                extra_body["sequential_image_generation"] = "disabled"
                info_log.append(f"已启用图改图模式,带 {len(data_uris)} 张参考图")

        if extra_body:
            std_kwargs["extra_body"] = extra_body

        # ----- 实际调用 -----
        response = client.images.generate(**std_kwargs)

        # ----- 解析返回 -----
        # 优先用 b64_json；若为空，降级到 url
        data0 = response.data[0]
        b64_str = getattr(data0, "b64_json", None)

        # 文件名带毫秒时间戳，避免并发冲突
        local_path = os.path.join(LOCAL_IMAGE_DIR, f"poster_{int(time.time()*1000)}.png")

        if b64_str:
            # 主路径：base64 解码并落盘
            img_bytes = base64.b64decode(b64_str)
            with open(local_path, "wb") as f:
                f.write(img_bytes)
            info_log.append(f"渲染成功:{len(img_bytes)//1024}KB")
            return local_path, " | ".join(info_log)

        # ----- 降级路径：b64_json 为空 → 尝试 url -----
        img_url = getattr(data0, "url", None)
        if img_url:
            info_log.append(f"⚠️ b64_json 为空,降级 URL 下载")
            try:
                resp = requests.get(img_url, timeout=30)
                if resp.status_code == 200:
                    with open(local_path, "wb") as f:
                        f.write(resp.content)
                    return local_path, " | ".join(info_log)
                return None, f"URL 下载失败: HTTP {resp.status_code}"
            except Exception as net_e:
                return None, f"URL 下载异常: {net_e}"

        return None, "API 既无 b64_json 也无 url 返回"

    except Exception as e:
        # ----- 异常详情透出 -----
        # 提取火山方舟服务端返回的具体错误码/信息，UI 直接看到根因
        err_type = type(e).__name__
        err_msg = str(e)
        detail = ""
        if hasattr(e, "response") and e.response is not None:
            try:
                detail = f" | HTTP {e.response.status_code} | {e.response.text[:300]}"
            except Exception:
                pass
        elif hasattr(e, "body") and e.body:
            try:
                detail = f" | body={str(e.body)[:300]}"
            except Exception:
                pass
        ref_hint = f" (本次调用携带了 {len(reference_images)} 张参考图)" if reference_images else ""
        return None, f"图像 API 异常 [{err_type}]: {err_msg}{detail}{ref_hint}"

print("✅ Visual Agent 核心 API 调用函数定义完成")

✅ Visual Agent 核心 API 调用函数定义完成


---

## 第十九步：MultiAgentSystem 类 — 主框架（__init__ 与 llm_chat）

下面三个步骤把多智能体系统封装成一个 `MultiAgentSystem` 类。

为了保持每个代码单元短小聚焦，类的方法**采用「先定义类骨架，后续单元用赋值方式追加方法」**的模式。这是 Jupyter Notebook 中常用的模块化技巧。

本步定义：

- 类 `__init__`：保存配置 + 初始化 OpenAI 客户端
- `llm_chat`：通用的文本对话方法（所有 Agent 共用）
- 类常量 `MAX_REDRAW = 2`：Visual Agent 最多重绘次数（首次 + 2 次重绘 = 总共 3 次）

In [19]:
# ============================================================
# MultiAgentSystem 类 —— 主框架
# 后续单元会用 「类名.方法名 = 函数」 的方式继续追加方法
# ============================================================

class MultiAgentSystem:
    """多智能体协同系统。

    职责: 统一管理 Planner / Copywriter / Art Director / Visual 四个 Agent，
          串联工具调用与 LLM 推理，对外暴露一个 process_chat 入口。
    """

    def __init__(self, api_key, base_url, text_model):
        # 任意一项为空时回退到全局默认值，避免传空导致 SDK 报错
        self.api_key    = api_key    if api_key    else DEFAULT_API_KEY
        self.base_url   = base_url   if base_url   else DEFAULT_BASE_URL
        self.text_model = text_model if text_model else TEXT_MODEL
        # 复用同一个 OpenAI 客户端，省去重复 TLS 握手成本
        self.client = OpenAI(api_key=self.api_key, base_url=self.base_url)
        # Visual Agent 最多重绘次数（首次 + 2 次重绘 = 共 3 次出图机会）
        self.MAX_REDRAW = 2

    def llm_chat(self, messages):
        """通用 LLM 文本对话方法，所有文本 Agent 共用。

        参数:
            messages: OpenAI 标准 messages 列表
        返回:
            str: 模型回复内容；异常时返回 "Mock 文本 / 请求失败: ..."
        """
        try:
            response = self.client.chat.completions.create(
                model=self.text_model,
                messages=messages
            )
            return response.choices[0].message.content
        except Exception as e:
            # 不抛异常，返回带前缀的字符串，让上层调用统一处理
            return f"Mock 文本 / 请求失败: {str(e)}"

print("✅ MultiAgentSystem 主框架定义完成（待追加 process_chat / _execute_agents）")

✅ MultiAgentSystem 主框架定义完成（待追加 process_chat / _execute_agents）


---

## 第二十步：对话处理方法 process_chat

`process_chat` 是面向 UI 的入口方法。它的职责：

1. 把用户输入加入 LLM 对话记忆（保留多轮上下文）
2. 调用 Planner Agent
3. 用正则识别 Planner 是否输出了 `[EXECUTE_WORKFLOW: ...]` 触发口令
4. 如果识别到 → 进入 `_execute_agents` 启动下游全流程
5. 如果没识别到 → 普通对话，等待用户下一轮输入

**特别处理：流式输出**

方法用 `yield` 而不是 `return`——每完成一步就向 UI 推送一次状态更新，让用户实时看到进度。

**特别处理：商品参考图注入**

如果用户上传了图，会在用户消息后追加一段「系统补充信息」，告知 Planner「记得在简报里标明用户有商品图」。

In [20]:
# ============================================================
# 给 MultiAgentSystem 追加 process_chat 方法
# 这是面向 UI 的对话入口
# ============================================================

def process_chat(self, user_text, chat_history, llm_memory, product_image=None):
    """对话入口（生成器，流式 yield UI 更新）。

    参数:
        user_text:      用户本次输入文本
        chat_history:   Gradio Chatbot 的对话历史（用于显示）
        llm_memory:     LLM 上下文记忆（用于多轮对话）
        product_image:  可选商品参考图本地路径

    yield:
        (chat_history, llm_memory, agent_log, copy_text, image_path)
    """
    # 首次对话时，把 PLANNER_PROMPT 作为 system 消息注入记忆
    if not llm_memory:
        llm_memory.append({"role": "system", "content": PLANNER_PROMPT})

    # display_text: 给 UI 显示的内容（不含 "系统补充"）
    # injected_text: 给 LLM 的内容（含 "系统补充"，包含商品图信息）
    display_text = user_text
    injected_text = user_text

    # 若用户上传了商品图，追加系统提示让 Planner 知晓
    if product_image:
        injected_text = (
            f"{user_text}\n\n[系统补充信息] 用户已上传商品实物参考图({product_image}),"
            f"后续生成的海报需要以该商品为画面主体,请在简报中明确告知下游。"
        )

    # 把用户输入推进记忆与显示历史
    llm_memory.append({"role": "user", "content": injected_text})
    chat_history.append({"role": "user",      "content": display_text})
    chat_history.append({"role": "assistant", "content": "正在进行思维链演算..."})
    yield chat_history, llm_memory, "正在进行思维链演算...", "", None

    # ----- 调用 Planner -----
    planner_reply = self.llm_chat(llm_memory)

    # ----- 用正则识别触发口令 -----
    match = re.search(r'\[EXECUTE_WORKFLOW:\s*(.*?)\]', planner_reply, re.DOTALL)

    if match:
        # ===== 触发了工作流 =====
        task_instruction = match.group(1).strip()
        # 给用户显示的回复需要剥掉触发口令本身
        display_reply = planner_reply.replace(match.group(0), "").strip()
        if not display_reply:
            display_reply = "需求逻辑通过验证,团队引擎重启,准备渲染海报!"

        # 推进记忆与显示
        llm_memory.append({"role": "assistant", "content": planner_reply})
        chat_history[-1] = {"role": "assistant", "content": display_reply}

        # 构造初始日志
        ref_log = f"🖼️  已携带商品参考图: {product_image}\n" if product_image else ""
        agent_log = (
            f"🧠 Planner: 需求逻辑通过验证,开始触发团队作业...\n{ref_log}"
            f"📝 系统指派(原文):\n{task_instruction}\n"
        )
        yield chat_history, llm_memory, agent_log, "", None

        # 进入下游 Agent 流水线（这是另一个生成器，用 yield from 透传）
        yield from self._execute_agents(
            task_instruction, chat_history, llm_memory, agent_log, product_image
        )
    else:
        # ===== 普通对话回合 =====
        llm_memory.append({"role": "assistant", "content": planner_reply})
        chat_history[-1] = {"role": "assistant", "content": planner_reply}
        yield chat_history, llm_memory, "🙋 等待进一步回复...", "", None


# 把上面定义的 process_chat 函数挂到类上
MultiAgentSystem.process_chat = process_chat
print("✅ process_chat 方法已挂载到 MultiAgentSystem")

✅ process_chat 方法已挂载到 MultiAgentSystem


---

## 第二十一步：智能体执行管线 _execute_agents

本步是整个系统的「**生产流水线**」——一旦 Planner 触发工作流，所有下游动作都按以下顺序自动跑完：

```
阶段 0: Planner 工具 → parse_brief_to_json（结构化简报）
阶段 1: Copywriter
        ├─ 工具1: keyword_research（行业热词）
        ├─ LLM: 写文案
        ├─ 工具2: copy_scorer（自评打分）
        └─ 若 < 70 分 → 自动重写一次
阶段 2: Art Director
        ├─ 工具1: color_palette_designer（配色）
        ├─ 工具2: composition_template（构图）
        ├─ LLM: 写主体/环境描述
        └─ 工具3: prompt_structurer（组装最终 Prompt）
阶段 3: Visual Agent（最多 3 次循环）
        ├─ call_seedream_api（出图）
        ├─ image_quality_check（自检）
        └─ 若 < 70 分 → Prompt 注入修正建议 → 重绘
```

每一步都用 `yield` 推送日志，UI 上能看到「Agent 思考过程」。

In [21]:
# ============================================================
# 给 MultiAgentSystem 追加 _execute_agents 方法
# 这是「Planner 触发工作流后」的全自动生产流水线
# ============================================================

def _execute_agents(self, task_instruction, chat_history, llm_memory,
                    agent_log, product_image=None):
    """下游 Agent 执行管线（生成器）。

    参数:
        task_instruction: Planner 输出的自然语言简报
        chat_history:     Gradio 显示的对话历史
        llm_memory:       LLM 多轮记忆
        agent_log:        实时拼接的流水线日志
        product_image:    商品参考图（决定是否启用图改图模式）
    """

    # ===========================================
    # 阶段 0: Planner 工具 → parse_brief_to_json
    # ===========================================
    agent_log += "\n\n🧰 [Planner 工具] 调用 parse_brief_to_json 把简报结构化..."
    yield chat_history, llm_memory, agent_log, "", None

    brief_json = parse_brief_to_json(task_instruction, self.client, self.text_model)
    agent_log += (
        f"\n   ✓ 解析完成: 品类={brief_json.get('product_category')}, "
        f"调性={brief_json.get('tone')}, 风格={brief_json.get('visual_style')}"
    )
    yield chat_history, llm_memory, agent_log, "", None

    # ===========================================
    # 阶段 1: Copywriter（工具加持的文案创作）
    # ===========================================
    agent_log += "\n\n✍️ Copywriter 启动..."
    yield chat_history, llm_memory, agent_log, "", None

    # ----- 工具调用 1: 卖点/热词检索 -----
    agent_log += (
        "\n   🧰 [Copywriter 工具1] keyword_research(品类="
        f"{brief_json.get('product_category', '通用')})..."
    )
    yield chat_history, llm_memory, agent_log, "", None

    kw_result = keyword_research(brief_json.get('product_category', '通用'))
    agent_log += (
        f"\n      ✓ 命中行业库'{kw_result['matched_category']}',"
        f"返回卖点词 {len(kw_result['卖点关键词'])} 条、"
        f"话术模板 {len(kw_result['热门话术模板'])} 条"
    )
    yield chat_history, llm_memory, agent_log, "", None

    # ----- LLM 基于工具输出创作文案 -----
    copy_messages = [
        {"role": "system", "content": COPYWRITER_PROMPT_TEMPLATE.format(
            keyword_research_result=json.dumps(kw_result, ensure_ascii=False, indent=2)
        )},
        {"role": "user", "content":
            f"创意简报:\n{task_instruction}\n\n结构化要点:\n"
            f"{json.dumps(brief_json, ensure_ascii=False, indent=2)}"}
    ]
    copywriter_result = self.llm_chat(copy_messages)

    # ----- 工具调用 2: 文案自评 -----
    agent_log += "\n   🧰 [Copywriter 工具2] copy_scorer 自评打分..."
    yield chat_history, llm_memory, agent_log, copywriter_result, None

    score_result = copy_scorer(copywriter_result, task_instruction)
    agent_log += (
        f"\n      ✓ 文案得分 {score_result['score']}/100 | "
        f"反馈:{'; '.join(score_result['feedback'])}"
    )
    yield chat_history, llm_memory, agent_log, copywriter_result, None

    # ----- 不及格 → 自动重写一次 -----
    if score_result['score'] < 70:
        agent_log += "\n   ↻ 分数偏低,触发自动重写..."
        yield chat_history, llm_memory, agent_log, copywriter_result, None

        rewrite_messages = copy_messages + [
            {"role": "assistant", "content": copywriter_result},
            {"role": "user", "content":
                f"自评反馈如下,请针对性改写:\n{score_result['feedback']}\n\n"
                f"再次输出三行格式(主标题/副标题/CTA)。"}
        ]
        copywriter_result = self.llm_chat(rewrite_messages)
        score_result = copy_scorer(copywriter_result, task_instruction)
        agent_log += f"\n      ✓ 改写后得分 {score_result['score']}/100"

    agent_log += "\n   ✅ Copywriter 文案交付!"
    yield chat_history, llm_memory, agent_log, copywriter_result, None

    # ===========================================
    # 阶段 2: Art Director（工具加持的视觉决策）
    # ===========================================
    agent_log += "\n\n🎨 Art Director 启动..."
    yield chat_history, llm_memory, agent_log, copywriter_result, None

    # 把视觉风格 + 文案调性合成「风格关键字」喂给配色/构图工具
    style_hint = f"{brief_json.get('visual_style', '')} {brief_json.get('tone', '')}"

    # ----- 工具调用 1: 配色 -----
    agent_log += f"\n   🧰 [Art Director 工具1] color_palette_designer(风格='{style_hint}')..."
    yield chat_history, llm_memory, agent_log, copywriter_result, None
    palette = color_palette_designer(style_hint)
    agent_log += (
        f"\n      ✓ 选取配色'{palette['name']}': "
        f"主色{palette['primary']}/辅色{palette['secondary']}/点缀{palette['accent']}"
    )
    yield chat_history, llm_memory, agent_log, copywriter_result, None

    # ----- 工具调用 2: 构图 -----
    agent_log += (
        "\n   🧰 [Art Director 工具2] composition_template(品类='"
        f"{brief_json.get('product_category', '')}')..."
    )
    yield chat_history, llm_memory, agent_log, copywriter_result, None
    composition = composition_template(brief_json.get('product_category', ''), style_hint)
    agent_log += f"\n      ✓ 推荐构图'{composition['name']}': {composition['fit_for']}"
    yield chat_history, llm_memory, agent_log, copywriter_result, None

    # ----- LLM 基于工具输出写主体/环境 -----
    ad_messages = [
        {"role": "system", "content": ART_DIRECTOR_PROMPT_TEMPLATE.format(
            palette_result=json.dumps(palette, ensure_ascii=False, indent=2),
            composition_result=json.dumps(composition, ensure_ascii=False, indent=2),
            copy_result=copywriter_result,
            has_reference_image="有(下游 Visual Agent 会传入商品实物作参考图)" if product_image else "无"
        )},
        {"role": "user", "content":
            f"创意简报结构化要点:\n{json.dumps(brief_json, ensure_ascii=False, indent=2)}"}
    ]
    subject_env = self.llm_chat(ad_messages)

    # 解析 LLM 返回的「主体/环境」两行
    subject_match = re.search(r'主体[::]\s*(.+)', subject_env)
    env_match     = re.search(r'环境[::]\s*(.+)', subject_env)
    subject     = subject_match.group(1).strip() if subject_match else subject_env
    environment = env_match.group(1).strip()     if env_match     else ""

    # ----- 工具调用 3: 结构化 Prompt 组装 -----
    agent_log += "\n   🧰 [Art Director 工具3] prompt_structurer 拼装最终 Prompt..."
    yield chat_history, llm_memory, agent_log, copywriter_result, None

    ad_prompt_result = prompt_structurer(
        subject=subject,
        environment=environment,
        palette=palette,
        composition=composition,
        copy_for_poster=copywriter_result
    )
    agent_log += f"\n   ✅ Art Director 完成最终 Prompt:\n----------\n{ad_prompt_result}\n----------"
    yield chat_history, llm_memory, agent_log, copywriter_result, None

    # ===========================================
    # 阶段 3: Visual Agent（出图 + 自检 + 重绘循环）
    # ===========================================
    current_prompt = ad_prompt_result
    image_path = None
    reference_images_list = [product_image] if product_image else None
    if product_image:
        agent_log += f"\n\n📸 Visual Agent 已启用「图改图模式」,商品参考图: {product_image}"

    # 最多重绘 MAX_REDRAW 次（首次 + 重绘 = 共 MAX_REDRAW+1 次出图机会）
    for attempt in range(self.MAX_REDRAW + 1):
        agent_log += f"\n\n📸 Visual Agent: 第 {attempt + 1} 次渲染..."
        yield chat_history, llm_memory, agent_log, copywriter_result, image_path

        # 出图
        image_path, info_msg = call_seedream_api(
            self.api_key, self.base_url,
            prompt=current_prompt,
            reference_images=reference_images_list
        )
        agent_log += f"\n   ℹ️ {info_msg}"

        if not image_path:
            # 出图直接失败 → 没有恢复手段，提前结束
            agent_log += "\n   ❌ 渲染失败,提前结束(详细错误见上一行)。"
            yield chat_history, llm_memory, agent_log, copywriter_result, None
            return

        # 自检
        agent_log += "\n   🧰 [Visual 工具] image_quality_check 自检..."
        yield chat_history, llm_memory, agent_log, copywriter_result, image_path
        qc = image_quality_check(image_path)
        agent_log += (
            f"\n      ✓ 自检得分 {qc['score']}/100 | 尺寸 {qc.get('size', '?')} "
            f"| 亮度 {qc.get('brightness', '?')}"
        )
        agent_log += f"\n      问题: {'; '.join(qc['issues'])}"

        if qc['pass']:
            # 自检通过 → 交付
            agent_log += "\n   ✅ 图像质检通过,交付!"
            yield chat_history, llm_memory, agent_log, copywriter_result, image_path
            return

        if attempt < self.MAX_REDRAW:
            # 把质检建议追加到 Prompt 末尾，重绘
            agent_log += f"\n   ↻ 质检未通过,根据建议改写 Prompt 重绘: {'; '.join(qc['suggestions'])}"
            current_prompt = current_prompt + "\n【修正补强】" + ";".join(qc['suggestions'])
            yield chat_history, llm_memory, agent_log, copywriter_result, image_path
        else:
            # 已达上限 → 即便不达标也交付当前最佳
            agent_log += "\n   ⚠️ 已达最大重绘次数,交付当前最佳版本。"
            yield chat_history, llm_memory, agent_log, copywriter_result, image_path


# 把上面定义的 _execute_agents 函数挂到类上
MultiAgentSystem._execute_agents = _execute_agents
print("✅ _execute_agents 方法已挂载到 MultiAgentSystem（多智能体系统已完整）")

✅ _execute_agents 方法已挂载到 MultiAgentSystem（多智能体系统已完整）


---

## 第二十二步：Gradio UI 界面搭建

用 Gradio 构建一个双栏 Web 界面：

**左栏（用户交互区）：**

- 聊天框 — 与 Planner 对话
- 输入框 — 多行文本框，回车即可发送
- 商品图上传 — 可选，启用「图改图」模式
- 发送/重置按钮

**右栏（输出展示区）：**

- 智能体协作日志 — 实时流式输出（可看到每一步工具调用）
- 最终文案展示
- 最终海报图展示

**`interact_with_flow` 函数**是 UI 与 `MultiAgentSystem` 之间的胶水：每次用户发送消息时实例化系统并把生成器的 yield 转发给 UI 组件。

In [22]:
# ============================================================
# Gradio UI 入口函数 + Blocks 界面定义
# ============================================================

def interact_with_flow(user_text, product_image, chat_history, llm_memory):
    """UI 与 MultiAgentSystem 之间的胶水函数（生成器）。

    每次用户发送消息时调用一次。把后端的流式 yield 透传给前端组件。
    """
    # 输入校验：空消息直接提示并退出
    if not user_text.strip():
        yield "", chat_history, llm_memory, "请输入需求", "", None
        return

    # 实例化系统（每次对话都用全局默认配置）
    system = MultiAgentSystem(DEFAULT_API_KEY, DEFAULT_BASE_URL, TEXT_MODEL)

    # 把后端流转发给前端：第一个返回值 "" 用于清空输入框
    for current_hist, current_mem, log, copy, img in system.process_chat(
        user_text, chat_history, llm_memory, product_image=product_image
    ):
        yield "", current_hist, current_mem, log, copy, img


# ----- Blocks 界面 -----
with gr.Blocks(title="AIGC 电商海报智能生成系统", theme=gr.themes.Soft()) as demo:
    # 顶部说明区
    gr.Markdown("""# ✦ AIGC 电商海报智能生成系统
> 由四大专业智能体协同驱动，从一句话需求到完整商业海报，全自动完成。

🧠 **Planner** Doubao-1.5-32k · 简报解析 · 对话记忆 · ReAct路由　　✍️ **Copywriter** Doubao-1.5-32k · 卖点热词库 · 文案打分 · 自动重写　　🎨 **Art Director** Doubao-1.5-32k · 配色库 · 构图模板 · Prompt组装　　📸 **Visual Agent** Doubao-Seedream-5.0-lite · 图像自检 · 自动重绘
""")

    with gr.Row():
        # ---------- 左栏：用户交互区 ----------
        with gr.Column(scale=1):
            chatbot = gr.Chatbot(label="🗨️ 与项目主控 Planner 对话", height=500)
            user_input = gr.Textbox(
                label="输入需求",
                placeholder="描述你的产品和海报需求，例如：帮我做一张夏日冷萃咖啡海报...",
                lines=2
            )
            product_image_in = gr.Image(
                label="🛍️ (可选) 上传商品实物图 — 模型将以此为主体融合进海报",
                type="filepath",
                height=180
            )
            with gr.Row():
                send_btn  = gr.Button("✉️ 发送指令", variant="primary")
                clear_btn = gr.Button("🗑️ 重新开始", variant="secondary")
            # State 用于在多次回调间持久化 LLM 上下文记忆
            llm_memory = gr.State([])

        # ---------- 右栏：输出展示区 ----------
        with gr.Column(scale=2):
            agent_log  = gr.Textbox(label="💡 智能体协作流水线 · 实时追踪",
                                    interactive=False, lines=18)
            final_copy = gr.Textbox(label="✨ Copywriter 输出文案",
                                    interactive=False, lines=4)
            final_img  = gr.Image(label="🖼️ Visual Agent 渲染海报", type="filepath")

    # ----- 事件绑定 -----
    # send_btn 点击 与 user_input 回车 共用同一份回调配置
    interact_args = {
        "fn": interact_with_flow,
        "inputs": [user_input, product_image_in, chatbot, llm_memory],
        "outputs": [user_input, chatbot, llm_memory, agent_log, final_copy, final_img]
    }
    send_btn.click(**interact_args)
    user_input.submit(**interact_args)

    # 重置按钮：把所有状态清空
    clear_btn.click(
        lambda: ([], [], "系统已重置，等待新指令...", "", None, None),
        inputs=None,
        outputs=[chatbot, llm_memory, agent_log, final_copy, final_img, product_image_in],
        queue=False
    )

print("✅ Gradio UI 定义完成，等待启动")

C:\Users\86139\AppData\Local\Temp\ipykernel_2340\4208124101.py:26: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="AIGC 电商海报智能生成系统", theme=gr.themes.Soft()) as demo:


✅ Gradio UI 定义完成，等待启动


---

## 第二十三步：启动应用

调用 `demo.launch()` 在本地起一个 Web 服务，浏览器自动打开 `http://127.0.0.1:7861`。

- `debug=True` — 控制台打印详细日志，便于排查
- `inbrowser=True` — 启动后自动用默认浏览器打开
- 外层 try/except — 极端情况下的兜底，避免 launch 参数兼容性问题导致整体崩溃

**使用建议：**

1. 第一次对话先用大白话描述需求，让 Planner 帮你引导
2. 如果有商品实物图，强烈建议上传——海报里就是你的真品
3. 不满意可以继续对话「换成XX风格」「文案再短一点」等，Planner 会重新触发流程
4. 想从头开始 → 点「🗑️ 重新开始」

In [ ]:
# ============================================================
# 启动 Gradio 服务
# 默认监听 http://127.0.0.1:7861（端口被占用时 Gradio 会自动 +1）
# ============================================================

try:
    # 主路径：自动打开浏览器
    demo.launch(debug=True, inbrowser=True)
except Exception:
    # 兜底：极端情况下重试一次（关闭外链共享，避免外网穿透问题）
    demo.launch(debug=True, inbrowser=True, share=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


C:\Users\86139\AppData\Local\Temp\ipykernel_2340\420375827.py:58: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  pixels = list(thumb.getdata())
C:\Users\86139\AppData\Local\Temp\ipykernel_2340\420375827.py:58: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  pixels = list(thumb.getdata())
C:\Users\86139\AppData\Local\Temp\ipykernel_2340\420375827.py:58: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  pixels = list(thumb.getdata())
C:\Users\86139\AppData\Local\Temp\ipykernel_2340\420375827.py:58: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  pixels = list(thumb.getdata())
C:\Users\86139\AppData\Local\Temp\ipykernel_2340\420375827.py:58: Deprec